In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
from DisRNN import MyDisRNN
from helper_functions import format_matrix, smooth
import matplotlib.pyplot as plt
import os
#from google.colab import drive
#drive.mount('/content/drive')

In [2]:
def format_matrix(M, name, row_prefix= 'rule', col_prefix= 'dim'):
    M = np.atleast_2d(M)
    n_rows, n_cols = M.shape

    header = '      ' + ' '.join(f'{col_prefix}{j:>2}' for j in range(n_cols))
    lines = [f'{name}:', header]
    for i, row in enumerate(M):
        row_str = ' '.join(f'{v:5.2f}' for v in row)
        lines.append(f'{row_prefix}{i:>2} | {row_str}')
    
    return '\n'.join(lines)

In [3]:
def build_update_MLP(input_size, output_size= 2, hidden_size= 5):
    return nn.Sequential(
        nn.Linear(input_size, hidden_size),
        nn.ReLU(),
        nn.Linear(hidden_size, hidden_size),
        nn.ReLU(),
        nn.Linear(hidden_size, hidden_size),
        nn.ReLU(),
        nn.Linear(hidden_size, output_size)
    )

def build_choice_MLP(input_size, output_size, hidden_size= 2):
    return nn.Sequential(
        nn.Linear(input_size, hidden_size),
        nn.ReLU(),
        nn.Linear(hidden_size, hidden_size),
        nn.ReLU(),
        nn.Linear(hidden_size, output_size)
    )

In [ ]:
class MyDisRNN(nn.Module):
    def __init__(self, m, n, q):
        super().__init__()

        self.m = m
        self.n = n
        self.q = q

        # build update MLPs and choice MLP
        self.updateMLPs = nn.ModuleList(
            [build_update_MLP(m+n) for _ in range(m)]
        )
        self.choiceMLP = build_choice_MLP(m,q)
            
        # initialize bottleneck parameters
        self.logit_M_h = nn.Parameter(5 * torch.ones((m,m)))  # M(i,j) -> update rule i's dependence on latent j
        self.log_sigma_h = nn.Parameter(-2 * torch.ones((m,m)))

        self.logit_M_x = nn.Parameter(5 * torch.ones((m,n)))
        self.log_sigma_x = nn.Parameter(-2 * torch.ones((m,n)))

        self.logit_M_z = nn.Parameter(5 *torch.ones(m))
        self.log_sigma_z = nn.Parameter(-2 * torch.ones(m))


    def bottleneck(self, x, m, sigma):
        if m.dim() == 1:
            noise = torch.randn_like(x) if self.training else torch.zeros_like(x)
            mean = m.unsqueeze(0) * x
            std = sigma.unsqueeze(0)
        else:
            shape = (m.shape[0], *x.shape)

            noise = torch.randn(shape, device= x.device) if self.training else torch.zeros(shape, device= x.device)
            mean = m.unsqueeze(1) * x.unsqueeze(0)
            std = sigma.unsqueeze(1)

        z = mean + std * noise
        kl = 0.5 * torch.sum(
            mean**2 + std**2 - torch.log(std**2 + 1e-8) - 1, 
            dim= -1
        )
        
        return z, kl

    def step(self, h, x):
        h_prev = h

        # bottlenecks for disentangled update rules
        M_h = torch.sigmoid(self.logit_M_h)
        sigma_h = torch.exp(self.log_sigma_h)
        h, kl_h = self.bottleneck(h, M_h, sigma_h)

        M_x = torch.sigmoid(self.logit_M_x)
        sigma_x = torch.exp(self.log_sigma_x)
        x, kl_x = self.bottleneck(x, M_x, sigma_x)

        # apply update MLPs
        z_outs = []
        for i, MLP in enumerate(self.updateMLPs):
            z_i = torch.cat([h[i], x[i]], dim= -1)
            logit_w, u = MLP(z_i).unbind(dim= -1)
            w = torch.sigmoid(logit_w)
            z_out = (1 - w) * h_prev[:, i] + w * u
            z_outs.append(z_out)
        z = torch.stack(z_outs, dim= -1)

        # bottlenecks for disentangled latents
        M_z = torch.sigmoid(self.logit_M_z)
        sigma_z = torch.exp(self.log_sigma_z)
        z, kl_z = self.bottleneck(z, M_z, sigma_z)

        kls = {
            'h': kl_h.sum(dim= 0),
            'x': kl_x.sum(dim= 0),
            'z': kl_z
        }

        return z, kls


    def out(self, h):
        return self.choiceMLP(h)


    def forward(self, X):
        T, B, n = X.shape
        assert n == self.n

        latents = []
        bottleneck_losses = {'h': [], 'x': [], 'z': []}
        outputs = []
        
        h = torch.zeros(B, self.m, device= X.device)
        latents.append(h)
        for t in range(T):
            x = X[t]

            h, kls = self.step(h, x)
            latents.append(h)
            for key, val in kls.items():
                bottleneck_losses[key].append(val)

            y = self.out(h)
            outputs.append(y)

        bottleneck_losses = {key: torch.stack(vals) for key, vals in bottleneck_losses.items()}

        return torch.stack(outputs), torch.stack(latents), bottleneck_losses

In [5]:
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

os.makedirs(f'checkpoints/seed{seed}', exist_ok= True)
os.makedirs(f'figs/seed{seed}', exist_ok= True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# define bandit task space
D = torch.rand
num_arms = 2


# initialize models and optimizers
input_size = 2

DisRNN_hidden_size = 16
DisRNN = MyDisRNN(DisRNN_hidden_size, input_size, num_arms).to(device)
DisRNN_critic = torch.nn.Linear(DisRNN_hidden_size, 1).to(device)
DisRNN_optimizer = torch.optim.Adam(
    list(DisRNN.parameters()) + list(DisRNN_critic.parameters()), 
    lr= 5e-3
)

LSTM_hidden_size = 48
LSTM = torch.nn.LSTM(input_size, LSTM_hidden_size).to(device)
LSTM_readout = torch.nn.Linear(LSTM_hidden_size, num_arms).to(device)
LSTM_critic = torch.nn.Linear(LSTM_hidden_size, 1).to(device)
LSTM_optimizer = torch.optim.Adam(
    list(LSTM.parameters()) + list(LSTM_readout.parameters()) + list(LSTM_critic.parameters()), 
    lr= 1e-3
)

In [ ]:
# training hyperparameters
episodes = 50_000
batch_size = 32
batch_idx = torch.arange(batch_size, device= device)
T = 100
beta_e = 1.0
anneal_end = 2000
beta_v = 0.05
beta_max = 1e-3
warmup_start = 0
warmup_end = 10_000

In [ ]:
# load checkpoint
episode = 49999
episodes = 60000
checkpoint = torch.load(f'checkpoints/seed{seed}/checkpoint_ep{episode}.pt')
# checkpoint = torch.load(f'/content/drive/MyDrive/checkpoints/seed{seed}/checkpoint_ep{episode}.pt')

DisRNN.load_state_dict(checkpoint['DisRNN_state_dict'])
DisRNN_critic.load_state_dict(checkpoint['DisRNN_critic_state_dict'])
DisRNN_optimizer.load_state_dict(checkpoint['DisRNN_optimizer_state_dict'])

LSTM.load_state_dict(checkpoint['LSTM_state_dict'])
LSTM_readout.load_state_dict(checkpoint['LSTM_readout_state_dict'])
LSTM_critic.load_state_dict(checkpoint['LSTM_critic_state_dict'])
LSTM_optimizer.load_state_dict(checkpoint['LSTM_optimizer_state_dict'])

In [ ]:
# training
DisRNN_regret_history = []
LSTM_regret_history = []
for ep in range(episodes):
    # sample task
    probs = D(batch_size, num_arms, device= device)


    # reset DisRNN state
    DisRNN.train()
    DisRNN_optimizer.zero_grad()

    DisRNN_h = torch.zeros(batch_size, DisRNN_hidden_size, device= device)
    DisRNN_x = torch.zeros(batch_size, input_size, device= device)

    DisRNN_log_probs = []
    DisRNN_rewards = []
    DisRNN_expected_rewards = []
    DisRNN_entropies = []
    DisRNN_bottleneck_losses = {'h': [], 'x': [], 'z': []}
    DisRNN_regrets = []

    # reset LSTM state
    LSTM.train()
    LSTM_optimizer.zero_grad()

    LSTM_h = torch.zeros(1, batch_size, LSTM_hidden_size, device= device)
    LSTM_c = torch.zeros(1, batch_size, LSTM_hidden_size, device= device)
    LSTM_x = torch.zeros(batch_size, input_size, device= device)

    LSTM_log_probs = []
    LSTM_rewards = []
    LSTM_expected_rewards = []
    LSTM_entropies = []
    LSTM_regrets = []


    for t in range(T):
        if t % 50 == 0:
            DisRNN_h = DisRNN_h.detach()
            LSTM_h = LSTM_h.detach()
            LSTM_c = LSTM_c.detach()


        # DisRNN step
        DisRNN_h, kls = DisRNN.step(DisRNN_h, DisRNN_x)
        DisRNN_logits = DisRNN.out(DisRNN_h)

        DisRNN_pi = torch.distributions.Categorical(logits= DisRNN_logits)
        DisRNN_a = DisRNN_pi.sample()
        DisRNN_r = (torch.rand(batch_size, device= device) < probs[batch_idx, DisRNN_a]).float()

        DisRNN_log_probs.append(DisRNN_pi.log_prob(DisRNN_a))
        DisRNN_rewards.append(DisRNN_r)
        DisRNN_expected_rewards.append(DisRNN_critic(DisRNN_h.detach()).squeeze(-1))
        DisRNN_entropies.append(DisRNN_pi.entropy())
        for key, val in kls.items():
            DisRNN_bottleneck_losses[key].append(val)
        DisRNN_regrets.append(probs.max(dim= -1).values - probs[batch_idx, DisRNN_a])

        DisRNN_x = torch.stack([2*DisRNN_a.float() - 1, 2*DisRNN_r - 1], dim= -1)


        # LSTM step
        LSTM_out, (LSTM_h, LSTM_c) = LSTM(LSTM_x.unsqueeze(0), (LSTM_h, LSTM_c))
        LSTM_logits = LSTM_readout(LSTM_out.squeeze(0))

        LSTM_pi = torch.distributions.Categorical(logits= LSTM_logits)
        LSTM_a = LSTM_pi.sample()
        LSTM_r = (torch.rand(batch_size, device= device) < probs[batch_idx, LSTM_a]).float()
        
        LSTM_log_probs.append(LSTM_pi.log_prob(LSTM_a))
        LSTM_rewards.append(LSTM_r)
        LSTM_expected_rewards.append(LSTM_critic(LSTM_out.squeeze(0).detach()).squeeze(-1))
        LSTM_entropies.append(LSTM_pi.entropy())
        LSTM_regrets.append(probs.max(dim= -1).values - probs[batch_idx, LSTM_a])

        LSTM_x = torch.stack([2*LSTM_a.float() - 1, 2*LSTM_r - 1], dim= -1)
        

    DisRNN_log_probs = torch.stack(DisRNN_log_probs)
    DisRNN_rewards = torch.stack(DisRNN_rewards)
    DisRNN_expected_rewards = torch.stack(DisRNN_expected_rewards)
    DisRNN_entropies = torch.stack(DisRNN_entropies)
    DisRNN_bottleneck_losses = {key: torch.stack(vals) for key, vals in DisRNN_bottleneck_losses.items()}
    DisRNN_regrets = torch.stack(DisRNN_regrets)
    DisRNN_regret_history.append(DisRNN_regrets.mean().item())

    LSTM_log_probs = torch.stack(LSTM_log_probs)
    LSTM_rewards = torch.stack(LSTM_rewards)
    LSTM_expected_rewards = torch.stack(LSTM_expected_rewards)
    LSTM_entropies = torch.stack(LSTM_entropies)
    LSTM_regrets = torch.stack(LSTM_regrets)
    LSTM_regret_history.append(LSTM_regrets.mean().item())


    # update betas
    beta_e = max(0.0, 1.0 - ep / anneal_end)
    beta = beta_max * min((ep - warmup_start) / (warmup_end - warmup_start), 1.0)
    
    # advantage actor-critic
    DisRNN_advantage = DisRNN_rewards - DisRNN_expected_rewards
    DisRNN_loss_actor = -(DisRNN_log_probs * DisRNN_advantage.detach()).mean()
    DisRNN_loss_critic = F.mse_loss(DisRNN_expected_rewards, DisRNN_rewards)
    DisRNN_loss_entropy = DisRNN_entropies.mean()
    DisRNN_loss_bottlenecks = sum(loss.mean() for loss in DisRNN_bottleneck_losses.values())
    DisRNN_loss = (
        DisRNN_loss_actor 
        + beta_v * DisRNN_loss_critic
        - beta_e * DisRNN_loss_entropy 
        + beta * DisRNN_loss_bottlenecks
    )
    
    DisRNN_loss.backward()
    torch.nn.utils.clip_grad_norm_(
        list(DisRNN.parameters()) + list(DisRNN_critic.parameters()),
        max_norm= 1.0
    )
    DisRNN_optimizer.step()

    LSTM_advantage = LSTM_rewards - LSTM_expected_rewards
    LSTM_loss_actor = -(LSTM_log_probs * LSTM_advantage.detach()).mean()
    LSTM_loss_critic = F.mse_loss(LSTM_expected_rewards, LSTM_rewards)
    LSTM_loss_entropy = LSTM_entropies.mean()
    LSTM_loss = (
        LSTM_loss_actor
        + beta_v * LSTM_loss_critic
        - beta_e * LSTM_loss_entropy
    )

    LSTM_loss.backward()
    torch.nn.utils.clip_grad_norm_(
        list(LSTM.parameters()) + list(LSTM_readout.parameters()) + list(LSTM_critic.parameters()),
        max_norm= 1.0
    )
    LSTM_optimizer.step()

    
    if ep % 250 == 0:
        DisRNN_avg_reward = DisRNN_rewards.mean().item()
        LSTM_avg_reward = LSTM_rewards.mean().item()
        print(f'ep {ep:5d}')
        print(f'LSTM avg r {LSTM_avg_reward:.3f} | DisRNN avg R {DisRNN_avg_reward:.3f}')
        M_h = torch.sigmoid(DisRNN.logit_M_h).detach().cpu().numpy()
        M_x = torch.sigmoid(DisRNN.logit_M_x).detach().cpu().numpy()
        M_z = torch.sigmoid(DisRNN.logit_M_z).detach().cpu().numpy()
        print()
        print(format_matrix(M_h, 'M_h', row_prefix= 'rule', col_prefix= 'lat'))
        print()
        print(format_matrix(M_x, 'M_x', row_prefix= 'rule', col_prefix= 'obs'))
        print()
        print(format_matrix(M_z.reshape(1,-1), 'M_z', row_prefix= 'lat', col_prefix= 'lat'))
        print()
        print()

    if (ep % 5000 == 0 and ep > 0) or ep == episodes - 1:
        torch.save({
            'ep': ep,
            'DisRNN_state_dict': DisRNN.state_dict(),
            'DisRNN_critic_state_dict': DisRNN_critic.state_dict(),
            'DisRNN_optimizer_state_dict': DisRNN_optimizer.state_dict(),
            'LSTM_state_dict': LSTM.state_dict(),
            'LSTM_readout_state_dict': LSTM_readout.state_dict(),
            'LSTM_critic_state_dict': LSTM_critic.state_dict(),
            'LSTM_optimizer_state_dict': LSTM_optimizer.state_dict(),
        }, f'checkpoints/seed{seed}/checkpoint_ep{ep}.pt') # f'/content/drive/MyDrive/checkpoints/seed{seed}/checkpoint_ep{ep}.pt')

        plt.figure(figsize= (8,5))
        plt.plot(smooth(np.array(DisRNN_regret_history)), label= 'DisRNN', color= 'green')
        plt.plot(smooth(np.array(LSTM_regret_history)), label= 'LSTM', color= 'black')
        plt.xlabel('Episode')
        plt.ylabel('Regret')
        plt.title('Model Regret Over Time')
        plt.legend()
        plt.savefig(f'figs/seed{seed}/regret_ep{ep}.png')
        # plt.savefig(f'/content/drive/MyDrive/figs/seed{seed}/regret_ep{ep}.png')
        plt.close()

In [ ]:
# load trained models
end_episode = episodes - 1
checkpoint = torch.load(f'checkpoints/seed{seed}/checkpoint_ep{end_episode}.pt')
# checkpoint = torch.load(f'/content/drive/MyDrive/checkpoints/seed{seed}/checkpoint_ep{episode}.pt')

DisRNN.load_state_dict(checkpoint['DisRNN_state_dict'])
DisRNN_critic.load_state_dict(checkpoint['DisRNN_critic_state_dict'])
DisRNN_optimizer.load_state_dict(checkpoint['DisRNN_optimizer_state_dict'])

LSTM.load_state_dict(checkpoint['LSTM_state_dict'])
LSTM_readout.load_state_dict(checkpoint['LSTM_readout_state_dict'])
LSTM_critic.load_state_dict(checkpoint['LSTM_critic_state_dict'])
LSTM_optimizer.load_state_dict(checkpoint['LSTM_optimizer_state_dict'])

In [ ]:
# testing
tests = 150

DisRNN_cumulative_regrets = []
LSTM_cumulative_regrets = []
for _ in range(tests):
    probs = D(batch_size, num_arms, device= device)

    # reset DisRNN state
    DisRNN.eval()

    DisRNN_h = torch.zeros(batch_size, DisRNN_hidden_size, device= device)
    DisRNN_x = torch.zeros(batch_size, input_size, device= device)

    # reset LSTM state
    LSTM.eval()

    LSTM_h = torch.zeros(1, batch_size, LSTM_hidden_size, device= device)
    LSTM_c = torch.zeros(1, batch_size, LSTM_hidden_size, device= device)
    LSTM_x = torch.zeros(batch_size, input_size, device= device)


    DisRNN_regrets = []
    LSTM_regrets = []
    with torch.no_grad():
        for t in range(T):
            # DisRNN step
            DisRNN_h, kls = DisRNN.step(DisRNN_h, DisRNN_x)
            DisRNN_logits = DisRNN.out(DisRNN_h)

            DisRNN_pi = torch.distributions.Categorical(logits= DisRNN_logits)
            DisRNN_a = DisRNN_pi.sample()
            DisRNN_r = (torch.rand(batch_size, device= device) < probs[batch_idx, DisRNN_a]).float()
            DisRNN_x = torch.stack([2*DisRNN_a.float() - 1, 2*DisRNN_r - 1], dim= -1)

            DisRNN_regrets.append(probs.max(dim= -1).values - probs[batch_idx, DisRNN_a])

            # LSTM step
            LSTM_out, (LSTM_h, LSTM_c) = LSTM(LSTM_x.unsqueeze(0), (LSTM_h, LSTM_c))
            LSTM_logits = LSTM_readout(LSTM_out.squeeze(0))

            LSTM_pi = torch.distributions.Categorical(logits= LSTM_logits)
            LSTM_a = LSTM_pi.sample()
            LSTM_r = (torch.rand(batch_size, device= device) < probs[batch_idx, LSTM_a]).float()
            LSTM_x = torch.stack([2*LSTM_a.float() - 1, 2*LSTM_r - 1], dim= -1)
            
            LSTM_regrets.append(probs.max(dim= -1).values - probs[batch_idx, LSTM_a])


            # drift
            probs += drift * torch.randn(batch_size, num_arms, device= device)
            probs = torch.clamp(probs, 0, 1)
            

    DisRNN_regrets = torch.stack(DisRNN_regrets)
    DisRNN_cumulative_regret = DisRNN_regrets.cumsum(dim= 0).mean(dim= 1).cpu().numpy()
    DisRNN_cumulative_regrets.append(DisRNN_cumulative_regret)
    
    LSTM_regrets = torch.stack(LSTM_regrets)
    LSTM_cumulative_regret = LSTM_regrets.cumsum(dim= 0).mean(dim= 1).cpu().numpy()
    LSTM_cumulative_regrets.append(LSTM_cumulative_regret)


DisRNN_mean = np.stack(DisRNN_cumulative_regrets).mean(axis= 0)
DisRNN_std = np.stack(DisRNN_cumulative_regrets).std(axis= 0)

LSTM_mean = np.stack(LSTM_cumulative_regrets).mean(axis= 0)
LSTM_std = np.stack(LSTM_cumulative_regrets).std(axis= 0)


plt.figure(figsize= (8,5))
plt.plot(DisRNN_mean, color= 'green', label= 'DisRNN')
plt.fill_between(range(T), DisRNN_mean - DisRNN_std, DisRNN_mean + DisRNN_std, alpha= 0.1, color= 'green')
plt.plot(LSTM_mean, color= 'black', label= 'LSTM')
plt.fill_between(range(T), LSTM_mean - LSTM_std, LSTM_mean + LSTM_std, alpha= 0.1, color= 'black')
plt.xlabel('Trial')
plt.ylabel('Cumulative Regret')
plt.title('Model Cumulative Regret')
plt.legend()
plt.grid()
plt.savefig(f'figs/seed{seed}/cumulative_regret.png')
# plt.savefig(f'/content/drive/MyDrive/figs/seed{seed}/cumulative_regret.png')
plt.close()